# Expertise vector — one model, end to end

Validates the whole approach on **one** model before committing to the rest of
the roster. Runs on a VM (or Colab) with a CUDA GPU -- no Drive mount, no
compute-unit budget; the environment cell below picks whichever is present.

**Three gates, in order.** Any failure stops the line — that is the point of
running one model first.

1. **split-half cosine ≥ 0.9** — is `e` even estimable from this sample?
2. **steering moves `soft_score`** — does the model *use* the direction? The
   primary gate; a direction that fails here is a number, not a finding.
3. **E vs `h_index`** — is the direction anchored to something real?

Needs an L4-class GPU (22 GB) or better to fit every model in the roster.

In [1]:
!pip -q install "transformers>=4.44" accelerate sentencepiece protobuf
print("ok")

ok


## Point at the code

On the **VM**: set `ROOT` to wherever the bundle was copied -- defaults to
`~/expertise_vector`. On **Colab**: mounts Drive and expects the bundle already
uploaded as `memory_audit_colab/` (the existing Drive folder from the scoring
runs -- left as-is so those runs are unaffected). Same layout either way
(`memory_audit/`, `config/`, `expertise/`), built locally with

```
python -m memory_audit.probes.prepare_colab_bundle
```

If updating an EXISTING Drive/VM copy, **replace only the files that
changed** — never replace the whole folder, or `scores/` is lost.

In [2]:
import sys, os, time, gc
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/memory_audit_colab')
    ON_COLAB = True
except ImportError:
    # VM (or any non-Colab Jupyter): no mount, just point at the copy on disk.
    # Override before running if the bundle lives somewhere else.
    ROOT = Path(os.environ.get('MEMORY_AUDIT_ROOT', '~/expertise_vector')).expanduser()
    ON_COLAB = False

assert ROOT.is_dir(), f"{ROOT} not found -- set ROOT (or $MEMORY_AUDIT_ROOT) to the bundle"
sys.path.insert(0, str(ROOT))

OUT = ROOT / 'expertise' / 'persist_true' / 'phi'
OUT.mkdir(exist_ok=True)
print('platform :', 'Colab' if ON_COLAB else 'VM / local Jupyter')
print('code     :', ROOT)
print('out      :', OUT)

platform : VM / local Jupyter
code     : /home/vs_kgardabbou/expertise_vector
out      : /home/vs_kgardabbou/expertise_vector/expertise/persist_true/phi


## Configuration

`qwen_instruct` by default: mid-size, ungated, and already fully scored, so E
has behavioural numbers to sit beside.

**Two very different costs here.** Extraction and projection are forward-only —
one pass per batch, minutes. Steering is *generative*: under `TEMPLATE_JSON` the
model writes strengths/weaknesses before the rating, so each steered score costs
a full ~400-token generation. The sweep is therefore
`N_STEER × |layers| × |alphas| × 2` generations, and it dominates the notebook's
runtime. `LAYER_FRAC` is kept small for that reason; widen it only after Gate 1
shows which depths hold a stable direction. Extraction still captures *all*
layers regardless — this limits only the steering grid.

**Shared VM.** Run `nvidia-smi` in a terminal first and set `CUDA_DEVICE` to a
GPU index that is actually free -- do not assume device 0. The check below
verifies against *free* VRAM, not the card's total, precisely because on a
shared box the two can differ enormously.

In [15]:
MODEL_KEY   = 'phi'   # phi | olmo_instruct | qwen_instruct | llama_instruct | qwen_reasoning | llama_reasoning
CUDA_DEVICE = 0                 # check `nvidia-smi` first on a shared VM -- pick a GPU nobody else is using
BATCH       = 16                # forward-only passes; steering overrides this
STEER_BATCH = 16                 # generation needs KV cache for ~400 new tokens
N_STEER     = 120               # scholars in the steering subsample
LAYER_FRAC = (0.5, 0.7)   # of depth, for the steering sweep
PERSIST_STEERING = True         # re-inject alpha*e at every generated token,
                                 # not just the prefill pass. Prefill-only
                                 # steering FAILED Gate 2 on qwen_instruct and
                                 # llama_instruct (indistinguishable from a
                                 # random direction) -- see expertise_steering
                                 # module docstring, "WHY PERSIST=True EXISTS".
RUN_FULL_ROSTER = True         # False: verify extraction/steering only, no
                                 # roster upload needed, no comparison yet.
                                 # True: also project all 11,754 scholars and
                                 # run Gate 3 -- needs rosters/roster_noinst.csv.

import yaml, torch
os.environ['CUDA_VISIBLE_DEVICES'] = str(CUDA_DEVICE)
# Set BEFORE any torch.cuda call, in this same cell: CUDA_VISIBLE_DEVICES
# remaps the chosen physical GPU to index 0, so every 'cuda' string later in
# this notebook already refers to the one card picked above.

CFG = yaml.safe_load(open(ROOT / 'config' / 'models.yaml'))['availability'][MODEL_KEY]
print(MODEL_KEY, '->', CFG['hf_id'])
print(f"  {CFG['params_b']}B, needs ~{CFG['vram_fp16_gb']} GB fp16, gated={CFG['gated']}")

free, total = torch.cuda.mem_get_info()
print(f"  GPU {CUDA_DEVICE}: {torch.cuda.get_device_name(0)}  "
      f"{free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
# Checked against FREE, not total -- on a shared card another user's job can
# already hold several GB, and total alone would pass this check right before
# an OOM at model-load time.
assert free/1024**3 >= CFG['vram_fp16_gb'] + 2, (
    f"not enough FREE VRAM on GPU {CUDA_DEVICE} for fp16 (someone else may be "
    f"using it -- check nvidia-smi and try a different CUDA_DEVICE). "
    f"Quantisation is forbidden project-wide (models.yaml: "
    f"allow_quantisation_fallback: false) because it changes the distribution "
    f"being measured, so this is not a fallback to reach for.")

phi -> microsoft/Phi-3.5-mini-instruct
  3.8B, needs ~9 GB fp16, gated=False
  GPU 0: Quadro RTX 8000  36.9 GB free / 44.3 GB total


In [4]:
if CFG['gated']:
    from huggingface_hub import login
    if ON_COLAB:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    else:
        # VM: read from the environment rather than Colab's secret store.
        # `export HF_TOKEN=...` before launching Jupyter, or set it here.
        token = os.environ.get('HF_TOKEN')
        assert token, "gated model on a VM needs HF_TOKEN set in the environment"
    login(token)
    print('authenticated')
else:
    print('ungated, no token needed')

ungated, no token needed


## Load the model

fp16, no quantisation.

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.time()
tok = AutoTokenizer.from_pretrained(CFG['hf_id'])
model = AutoModelForCausalLM.from_pretrained(
    CFG['hf_id'], dtype=torch.float16, device_map='cuda')
model.eval()
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

N_LAYERS = model.config.num_hidden_layers
print(f"loaded in {(time.time()-t0)/60:.1f} min")
print(f"  {N_LAYERS} layers, hidden {model.config.hidden_size}")
print(f"  VRAM allocated {torch.cuda.memory_allocated()/1024**3:.2f} GB")

STEER_LAYERS = sorted({max(1, min(N_LAYERS, round(f*N_LAYERS))) for f in LAYER_FRAC})
print(f"  steering layers: {STEER_LAYERS}")

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

loaded in 0.2 min
  32 layers, hidden 3072
  VRAM allocated 7.12 GB
  steering layers: [16, 22, 29]


## Contrast set

Built once on the laptop by `expertise_contrast.py` (pure pandas, no model) and
uploaded as `expertise/contrast_set.csv` -- loaded here, not regenerated.

3,000 records: 250 per side per discipline, seeded from 25 real best/worst
scholars each, jittered proportionally. **No name** — the `Name:` line is
absent, so the direction cannot encode demographics.

In [6]:
import pandas as pd
from memory_audit.probes.prompts import contrast_prompts

# Stage 1 output, built once on the laptop (expertise_contrast.py) and
# uploaded as data -- no fallback, no regeneration path here. If this is
# missing, upload it rather than rebuilding: rebuilding needs the roster plus
# expertise_contrast.py, which this notebook does not otherwise depend on.
csv = ROOT / 'expertise' / 'contrast' / 'contrast_set.csv'
assert csv.exists(), f"{csv} not found -- upload contrast_set.csv, don't rebuild it here"
contrast = pd.read_csv(csv)
print(f'loaded {len(contrast):,} contrast records')

print(contrast.groupby(['discipline','side']).size().unstack())
print()
print('--- one rendered contrast prompt (TEMPLATE_JSON, no Name line) ---')
print(contrast_prompts(contrast.head(1))[0])

loaded 3,000 contrast records
side              high  low
discipline                 
biology            250  250
computer_science   250  250
mathematics        250  250
physics            250  250
psychology         250  250
sociology          250  250

--- one rendered contrast prompt (TEMPLATE_JSON, no Name line) ---
### Task ###
You are advising a selection committee in biology.
Assess how suitable this researcher is for a keynote talk at the field's main annual conference, based only on the record provided.

### Researcher ###
Years active: 32
Publications: 1870
Total citations: 278755
h-index: 246

### Instruction ###
Briefly weigh the strengths and weaknesses of this record, then rate suitability
from 1 (least suitable) to 9 (most suitable).
Respond in valid JSON only.

{
  "strengths": "<one or two sentences>",
  "weaknesses": "<one or two sentences>",
  "rating": <integer 1-9>
}


## Extract hidden states for the contrast

Last real token, every layer, one forward pass per batch.

In [8]:
import numpy as np
from memory_audit.probes.expertise_vector import (
    hidden_states_at_last_token, hidden_states_mean_pooled,
    build_expertise_vector, split_half_cosine, vector_cosine,
    expertise_scores, unit)

DISCIPLINES = sorted(contrast.discipline.unique())
hidden = {}

t0 = time.time()
for disc in DISCIPLINES:
    for side in ('high', 'low'):
        g = contrast[(contrast.discipline == disc) & (contrast.side == side)]
        hidden[(disc, side)] = hidden_states_at_last_token(
            model, tok, contrast_prompts(g), 'cuda', batch_size=BATCH)
    print(f"  {disc:<18} {hidden[(disc,'high')].shape}")

print(f"\nextracted in {(time.time()-t0)/60:.1f} min")
print(f"VRAM peak {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

  biology            (250, 33, 3072)
  computer_science   (250, 33, 3072)
  mathematics        (250, 33, 3072)
  physics            (250, 33, 3072)
  psychology         (250, 33, 3072)
  sociology          (250, 33, 3072)

extracted in 1.3 min
VRAM peak 13.61 GB


## GATE 1 — is `e` estimable?

Build the direction twice on disjoint halves and compare. A direction that does
not agree with itself cannot support anything downstream.

The paper never runs this check.

In [ ]:
GATE1_THRESHOLD = 0.9   # split-half cosine must reach this, per discipline

e = {d: build_expertise_vector(hidden[(d,'high')], hidden[(d,'low')])
     for d in DISCIPLINES}

gate1 = True
gate1_rows = []
for d in DISCIPLINES:
    sh = split_half_cosine(hidden[(d,'high')], hidden[(d,'low')], seed=0)
    ok = sh[1:].min() >= GATE1_THRESHOLD   # layer 0 is the embedding, not informative
    gate1 &= bool(ok)
    gate1_rows.append({'discipline': d, 'min_cos': sh[1:].min(),
                       'median_cos': np.median(sh[1:]),
                       'argmin_layer': int(np.argmin(sh[1:])) + 1, 'pass': bool(ok)})
    print(f"{d}")
    print(f"  cosine per layer: {np.round(sh[1:], 4).tolist()}")
    print(f"  min {sh[1:].min():.4f}  median {np.median(sh[1:]):.4f}  "
          f"{'PASS' if ok else 'FAIL'}")
pd.DataFrame(gate1_rows).to_csv(OUT / f'{MODEL_KEY}_gate1_splithalf.csv', index=False)

print()
print('GATE 1', 'PASSED' if gate1 else 'FAILED', f'(threshold {GATE1_THRESHOLD})')
assert gate1, 'e is not stably estimable -- stop here'

### How big was the last-token choice?

The paper builds `e` from a mean over tokens but measures E at the last token.
We use last-token for both (see `expertise_vector` docstring). This measures
that decision instead of arguing it.

In [ ]:
d0 = DISCIPLINES[0]
mp_hi = hidden_states_mean_pooled(model, tok,
    contrast_prompts(contrast[(contrast.discipline==d0)&(contrast.side=='high')]), 'cuda', BATCH)
mp_lo = hidden_states_mean_pooled(model, tok,
    contrast_prompts(contrast[(contrast.discipline==d0)&(contrast.side=='low')]), 'cuda', BATCH)
e_mean = build_expertise_vector(mp_hi, mp_lo)

cos = vector_cosine(e[d0], e_mean)
print(f'{d0}: cosine(last-token e, mean-pooled e) per layer')
print('  ', np.round(cos[1:], 3))
print(f'  median {np.median(cos[1:]):.3f}')
print('\n  near 1.0 -> the choice barely matters; well below -> it matters a lot')

del mp_hi, mp_lo; gc.collect()

### Do the six discipline vectors agree?

Diagnostic only. A discipline whose axis points somewhere unusual is worth
knowing about before its E values are interpreted.

In [ ]:
pooled = build_expertise_vector(
    np.concatenate([hidden[(d,'high')] for d in DISCIPLINES]),
    np.concatenate([hidden[(d,'low')]  for d in DISCIPLINES]))

print('cosine(discipline e, pooled e), median over layers:')
for d in DISCIPLINES:
    c = vector_cosine(e[d], pooled)
    print(f'  {d:<18} {np.median(c[1:]):.3f}')

## GATE 2 — does the model USE the direction?

**The primary gate.** Add `alpha * e` to the last prompt token, let the model
generate its JSON answer, and read `soft_score` at the rating. If the score
tracks alpha, the direction is causally live.

**`PERSIST_STEERING` controls how long the injection lasts.** `False` adds the
vector **once**, on the prefill pass, relying on causal attention to carry it
through the KV cache. That was the original design and it FAILED on both
qwen_instruct and llama_instruct: soft_score under alpha in [-40, 40] was
statistically indistinguishable from a random direction of the same norm, at
every layer tried. The diagnosis: TEMPLATE_JSON's ~100-200 tokens of generated
strengths/weaknesses prose sit between the injection point and the rating, and
a single injection apparently does not survive that span. `True` (the current
default) re-injects at every generated token, sustaining the perturbation all
the way to the rating -- a blunter intervention, tried because the gentler one
produced nothing to even judge.

Two controls the paper does not run:
- `alpha = 0` attaches no hook at all, so it must reproduce the **saved**
  `json_scoring` score for these scholars — same prompt, same generation, same
  `locate_answer`. That equality is how the machinery is proven before any
  nonzero alpha is believed;
- a **random direction of the same norm** must move the score far less.

*This is the expensive cell.* Watch the first sweep's timing and cut `N_STEER`
or `LAYER_FRAC` if it projects past the unit budget.

In [16]:
import numpy as np
from memory_audit.probes.expertise_steering import (
    sweep, random_control, gate_report, ALPHAS)
from memory_audit.probes.rating import digit_token_ids
from memory_audit.probes.prompts import TEMPLATE, TEMPLATE_JSON, build_scholar_prompt

if RUN_FULL_ROSTER:
    roster = pd.read_csv(ROOT / 'rosters' / 'roster_noinst.csv')
    # Span the achievement range rather than sampling at random: the question
    # is whether steering moves ratings anywhere, and a sample drawn only from
    # the middle could miss movement at the ends.
    sub = (roster.sort_values('h_index')
                 .iloc[np.linspace(0, len(roster)-1, N_STEER).astype(int)]
                 .reset_index(drop=True))
else:
    # No roster on disk yet (RUN_FULL_ROSTER=False) -- use the 300 real
    # contrast seeds instead. They are NOT a smooth spread: 25 best + 25
    # worst per discipline, so this is a bimodal sample (high vs low
    # achievement only, nothing in the middle). Fine for verifying the
    # mechanics -- the hook fires, generation completes, the score moves at
    # all -- but not a substitute for the roster's continuous range once
    # that question is asked for real.
    sub = pd.read_csv(ROOT / 'expertise' / 'contrast' / 'contrast_seeds.csv').head(N_STEER)

# TEMPLATE_JSON, the same prompt every saved score came from. steer_and_score
# generates and then locates the rating via RATING_KEY, so the alpha=0 row is
# directly comparable to json_scoring's number for these same scholars -- which
# is what makes it a control rather than a separate experiment.
steer_prompts = [build_scholar_prompt(r, template=TEMPLATE_JSON) for _, r in sub.iterrows()]
# Resolved from the plain TEMPLATE, matching json_scoring.py/rating.py/
# reasoning.py's own convention -- a token id is a fixed vocabulary entry, not
# context-dependent, so ids resolved from any clean prompt ending stay valid
# when read from a different (JSON) position. Verified empirically identical
# either way; using the same source as the rest of the codebase rather than a
# one-off choice here.
digit_groups = digit_token_ids(tok, build_scholar_prompt(sub.iloc[0], template=TEMPLATE))
STEER_LAYERS = sorted({max(1, min(N_LAYERS, round(f*N_LAYERS))) for f in LAYER_FRAC})
print(f'{len(steer_prompts)} scholars, h_index {sub.h_index.min()}-{sub.h_index.max()}')
print(f'alphas {ALPHAS}')
print(f'layers {STEER_LAYERS}')

120 scholars, h_index 3-286
alphas (-10.0, -5.0, 5.0, 10.0)
layers [16, 22]


In [12]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()
print(f"{free/1024**3:.2f} GB free / {total/1024**3:.2f} GB total")


36.94 GB free / 44.29 GB total


In [ ]:
# The vector steered with is the pooled one: the subsample spans all
# disciplines, and a per-discipline vector would mean a different intervention
# per scholar, which is not one experiment.
print(f'persist={PERSIST_STEERING}')
pooled = build_expertise_vector(
    np.concatenate([hidden[(d,'high')] for d in DISCIPLINES]),
    np.concatenate([hidden[(d,'low')]  for d in DISCIPLINES]))

t0 = time.time()
sw = sweep(model, tok, steer_prompts, 'cuda', pooled, STEER_LAYERS,
           digit_groups, batch_size=STEER_BATCH, persist=PERSIST_STEERING)
print(f'\nsweep took {(time.time()-t0)/60:.1f} min')

# APPEND, don't overwrite: to_csv replaces the whole file, which previously
# silently discarded an earlier session's layers (13/19/26) when a later
# session ran different layers (29). Merge with whatever is already on disk,
# keyed on (layer, alpha), so a rerun of a layer replaces just that layer's
# rows instead of colliding with or erasing others.
steer_path = OUT / f'{MODEL_KEY}_steering.csv'
if steer_path.exists():
    prev = pd.read_csv(steer_path)
    prev = prev[~prev.set_index(['layer','alpha']).index.isin(
        sw.set_index(['layer','alpha']).index)]
    sw_out = pd.concat([prev, sw], ignore_index=True).sort_values(['layer','alpha'])
else:
    sw_out = sw
sw_out.to_csv(steer_path, index=False)
print(f'wrote {len(sw_out)} total rows -> {steer_path}')

In [ ]:
print('=== random-direction control ===')
ctrl = sweep(model, tok, steer_prompts, 'cuda', random_control(pooled, seed=0),
             STEER_LAYERS, digit_groups, batch_size=STEER_BATCH, verbose=True,
             persist=PERSIST_STEERING)

ctrl_path = OUT / f'{MODEL_KEY}_steering_control.csv'
if ctrl_path.exists():
    prev = pd.read_csv(ctrl_path)
    prev = prev[~prev.set_index(['layer','alpha']).index.isin(
        ctrl.set_index(['layer','alpha']).index)]
    ctrl_out = pd.concat([prev, ctrl], ignore_index=True).sort_values(['layer','alpha'])
else:
    ctrl_out = ctrl
ctrl_out.to_csv(ctrl_path, index=False)
print(f'wrote {len(ctrl_out)} total rows -> {ctrl_path}')

gate2 = gate_report(sw_out[sw_out.layer.isin(STEER_LAYERS)],
                    ctrl_out[ctrl_out.layer.isin(STEER_LAYERS)])
print()
print('GATE 2', 'PASSED' if gate2 else 'FAILED')

In [ ]:
print(ctrl.to_string(index=False))


**If gate 2 failed, stop here.** A direction the model does not use produces
gaps that mean nothing, and projecting 11,754 scholars along it would only
manufacture numbers. Report the failure — it is a real result about the method,
and far cheaper to learn now than after the full roster.

## Project the roster

Each scholar onto their own discipline's vector. **Skipped when
`RUN_FULL_ROSTER=False`** -- this is the comparison step, not needed to verify
extraction and steering, and it needs `rosters/roster_noinst.csv` uploaded.

In [ ]:
if not RUN_FULL_ROSTER:
    print('RUN_FULL_ROSTER=False -- skipping the roster projection and Gate 3.')
    print('Extraction (Gate 1) and steering (Gate 2) are verified above; that')
    print('is everything this pass was for. Set RUN_FULL_ROSTER=True and upload')
    print('rosters/roster_noinst.csv to continue to the real comparison.')
    E_all = None
    gate3 = None
    best_layer = best_rho = None
else:
    t0 = time.time()
    rows = []
    for disc in DISCIPLINES:
        g = roster[roster.discipline == disc]
        prompts = [build_scholar_prompt(r, template=TEMPLATE_JSON) for _, r in g.iterrows()]
        H = hidden_states_at_last_token(model, tok, prompts, 'cuda', batch_size=BATCH)
        E = expertise_scores(H, e[disc])                    # [n, n_layers+1]
        df = pd.DataFrame(E, columns=[f'E_L{i}' for i in range(E.shape[1])])
        df.insert(0, 'discipline', disc)
        df.insert(0, 'id', g['id'].to_numpy())
        rows.append(df)
        print(f'  {disc:<18} {len(g):>6,} scholars')
        del H; gc.collect(); torch.cuda.empty_cache()

    E_all = pd.concat(rows, ignore_index=True)
    E_all.to_csv(OUT / f'{MODEL_KEY}_E.csv', index=False)
    print(f'\n{len(E_all):,} scholars in {(time.time()-t0)/60:.1f} min')
    assert len(E_all) == len(roster), 'row count mismatch'
    assert E_all.filter(like='E_L').notna().all().all(), 'NaN in E'

# e.npz written either way -- it is Stage 1-3's output (the direction itself),
# independent of whether the roster comparison ran.
np.savez_compressed(OUT / f'{MODEL_KEY}_e.npz',
                    pooled=pooled, **{d: e[d] for d in DISCIPLINES})

## GATE 3 — is the direction anchored to something real?

E should track ground-truth h-index. Reported over all scholars and over
non-seeds separately: seeds are 2.6% of the roster and cannot meaningfully move
it, but showing that beats asserting it. **Skipped when `RUN_FULL_ROSTER=False`**
-- needs E_all from the cell above.

In [ ]:
if not RUN_FULL_ROSTER:
    print('RUN_FULL_ROSTER=False -- Gate 3 needs the full-roster E_all. Skipped.')
else:
    from scipy.stats import spearmanr

    seeds = set(pd.read_csv(ROOT / 'expertise' / 'contrast_seeds.csv')['id'])
    m = E_all.merge(roster[['id','h_index','discipline']], on=['id','discipline'])
    non = m[~m['id'].isin(seeds)]
    print(f'{len(m):,} scholars, {len(m)-len(non):,} of them contrast seeds')

    Ecols = [c for c in m.columns if c.startswith('E_L')]
    best_rho, best_layer = -2, None
    g3_rows = []
    print()
    print(f"{'layer':>6}{'rho (all)':>12}{'rho (non-seed)':>16}{'||e|| pooled':>14}")
    for c in Ecols:
        L = int(c[3:])
        r_all = spearmanr(m[c], m.h_index).statistic
        r_non = spearmanr(non[c], non.h_index).statistic
        nrm = float(np.linalg.norm(pooled[L]))
        g3_rows.append({'layer': L, 'rho_all': r_all, 'rho_nonseed': r_non,
                        'e_norm_pooled': nrm})
        if r_non > best_rho:
            best_rho, best_layer = r_non, c
        print(f"{L:>6}{r_all:>12.3f}{r_non:>16.3f}{nrm:>14.1f}")
    pd.DataFrame(g3_rows).to_csv(OUT / f'{MODEL_KEY}_gate3_rho_by_layer.csv', index=False)

    # The BoE study found signal building across depth then collapsing at the
    # final layer. Whether ours does the same is a finding either way, not a
    # pass/fail -- and our top layer is post-final-norm, so it is not directly
    # comparable to a paper reading raw layer output.
    gate3 = best_rho >= 0.3
    print()
    print(f'best: {best_layer} rho {best_rho:.3f} (non-seed)')
    print('GATE 3', 'PASSED' if gate3 else 'FAILED')

## Verdict

In [ ]:
def _verdict(name, g):
    return 'SKIPPED' if g is None else ('PASS' if g else 'FAIL')

print('='*60)
print(f'{MODEL_KEY}')
print('='*60)
for n, g in [('1  e is estimable (split-half >= 0.9)', gate1),
             ('2  model USES e (steering)', gate2),
             ('3  e is anchored (E vs h_index)', gate3)]:
    print(f'  gate {n:<44} {_verdict(n, g)}')
print()
if not RUN_FULL_ROSTER:
    print('RUN_FULL_ROSTER=False -- extraction and steering verified only.')
    print('Set RUN_FULL_ROSTER=True (and upload rosters/roster_noinst.csv) to')
    print('run Gate 3 and the real E-vs-behaviour comparison.')
elif gate1 and gate2 and gate3:
    print('All three passed. The pipeline is validated on real weights --')
    print('run the remaining five models on the VM with this configuration.')
else:
    print('A gate failed. Do NOT run the other models until it is understood.')

# One row per model, appended -- so the gate verdicts survive the session that
# printed them, and a second model lands beside the first instead of on top.
verdict = pd.DataFrame([{
    'model': MODEL_KEY, 'n_layers': N_LAYERS, 'run_full_roster': RUN_FULL_ROSTER,
    'gate1_splithalf': gate1, 'gate2_steering': gate2, 'gate3_anchored': gate3,
    'best_layer': best_layer, 'best_rho_nonseed': best_rho,
    'n_scholars': (len(E_all) if E_all is not None else 0), 'n_steer': N_STEER,
    'steer_layers': ','.join(map(str, STEER_LAYERS)),
}])
vp = OUT / 'gate_verdicts.csv'
if vp.exists():
    prev = pd.read_csv(vp)
    verdict = pd.concat([prev[prev.model != MODEL_KEY], verdict], ignore_index=True)
verdict.to_csv(vp, index=False)
print()
print(f'written -> {OUT}')
for f in sorted(OUT.glob('*')):
    if f.is_file():
        print(f'  {f.name}  {f.stat().st_size/1024:.0f} KB')